# Ultra-Scale Playbook 训练系统 · 第 1/14 课

> 状态：**学习中（待提交）**  
> 一次只完成一课；未通过前不要打开下一课答案。

## 统一完成标准

代码 4 分、Q1～Q3 各 2 分，通过线 8/10。必须解释正确性边界、显存/通信公式中的单位与分片维度；未实际运行的内容只能标记为静态审查。

# 第 1 课：训练流程与显存账本

- 对应官方章节：First Steps: Training on One GPU → Memory usage in transformers（静态部分）、附录 A2
- 前置：Python 基础、神经网络训练的大致概念
- 状态：学习中（代码工作簿 `ch1.ipynb`）

## 本课目标

完成后你需要能够：

- 解释一次训练迭代的三个阶段。
- 区分四类主要显存占用。
- 根据参数量和精度估算静态训练显存。
- 说明估算结果的边界，避免把理论值冒充峰值显存。

## 核心概念

一次普通训练迭代包括：

1. Forward：计算预测，并保存反向传播需要的激活。
2. Backward：利用激活计算梯度，逐步释放部分激活。
3. Optimizer step：使用梯度和优化器状态更新参数。

训练显存的四个主要部分是：


In [ ]:
参数 + 梯度 + 优化器状态 + 激活


前三项主要取决于参数量和精度，称为本课的"静态账本"。激活还取决于 batch size、序列长度和模型结构，将在第 2 课处理。

设模型有 \(N\) 个参数。

普通 FP32 + Adam：


In [ ]:
参数：4N bytes
梯度：4N bytes
Adam 一阶、二阶状态：8N bytes
合计：16N bytes


典型 BF16 混合精度 + Adam：


In [ ]:
BF16 参数：2N
BF16 梯度：2N
FP32 master weights：4N
FP32 Adam 状态：8N
合计：16N bytes


如果另外保留 FP32 梯度累积缓冲区，再增加 \(4N\) bytes，总计 \(20N\)。

这说明一个常见面试误区：**混合精度不一定降低参数、梯度和优化器状态的总显存。** 它的重要优势还包括更快的低精度计算和更小的激活。

这些估算不包含：

- 激活
- CUDA context 和内核开销
- 临时缓冲区
- 显存碎片
- 通信缓冲区

因此它是下界或组成分析，不是准确的峰值显存预测。

## 具体演示

一个 7B 参数模型，在不保留 FP32 梯度缓冲区时：


In [ ]:
7 × 10⁹ parameters × 16 bytes/parameter
= 112 × 10⁹ bytes
= 112 GB


仅静态状态就超过 80 GB，还没有计算激活。但严谨的结论应当是：

> 标准、未切分、未卸载的这种训练配置无法放进一张 80 GB GPU。

不能扩大成"7B 模型绝对无法在一张 80 GB GPU 上训练"，因为优化器、量化、分片、卸载和冻结参数都可能改变前提。

## 代码填空题

补全以下程序。不得删除某个显存分量，也不要直接写死最终总数。


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class MemoryBreakdown:
    """所有字段的单位都是 byte。"""

    model_bf16: int
    grad_bf16: int
    master_weights_fp32: int
    adam_m_fp32: int
    adam_v_fp32: int
    grad_accum_fp32: int

    @property
    def total(self) -> int:
        # 不能手写"每参数共多少 byte"；
        # 应从各分量求和，以便检查每一项是否被重复或遗漏。
        return ______________________________


def estimate_adam_memory(
    num_params: int,
    use_fp32_grad_accum: bool,
) -> MemoryBreakdown:
    """
    估算 BF16 混合精度训练的静态显存。

    假设：
    1. 前向、反向使用 BF16 参数和梯度。
    2. 保留 FP32 master weights。
    3. Adam 的一阶、二阶状态均为 FP32。
    4. 可选一个额外的 FP32 梯度累积缓冲区。
    5. 不包含激活、临时 buffer、CUDA context 和碎片。
    """
    if num_params <= 0:
        raise ValueError("num_params must be positive")

    # 每个 BF16 数值占多少 byte？
    bf16_bytes = ______

    # 每个 FP32 数值占多少 byte？
    fp32_bytes = ______

    model_bf16 = num_params * ______
    grad_bf16 = num_params * ______

    # master weights 是用于稳定更新的 FP32 参数副本。
    master_weights_fp32 = num_params * ______

    # Adam 为每个参数分别维护一阶矩和二阶矩。
    adam_m_fp32 = num_params * ______
    adam_v_fp32 = num_params * ______

    # 注意：这是额外缓冲区，不能拿它替代 BF16 梯度。
    grad_accum_fp32 = (
        num_params * ______ if use_fp32_grad_accum else 0
    )

    return MemoryBreakdown(
        model_bf16=model_bf16,
        grad_bf16=grad_bf16,
        master_weights_fp32=master_weights_fp32,
        adam_m_fp32=adam_m_fp32,
        adam_v_fp32=adam_v_fp32,
        grad_accum_fp32=grad_accum_fp32,
    )


def bytes_to_gb(num_bytes: int) -> float:
    """十进制 GB，硬件和原文表格常使用这个单位。"""
    return num_bytes / __________________


def bytes_to_gib(num_bytes: int) -> float:
    """二进制 GiB；1 GiB = 1024^3 bytes。"""
    return num_bytes / __________________


for size in (1_000_000_000, 7_000_000_000):
    for fp32_accum in (False, True):
        result = estimate_adam_memory(size, fp32_accum)
        print(
            f"params={size / 1e9:.0f}B, "
            f"fp32_grad_accum={fp32_accum}, "
            f"{bytes_to_gb(result.total):.1f} GB, "
            f"{bytes_to_gib(result.total):.1f} GiB"
        )


## 三个问答题


### Q1

为什么推理一个 7B BF16 模型可能只需约 14 GB 参数显存，而使用 Adam 训练时静态状态可能达到 112 GB？请逐项说明多出来的内容。


### Q2

BF16 混合精度在本课账本中可能仍是每参数 16 bytes。既然静态总量没有下降，为什么训练系统仍广泛使用它？至少说明两个收益，并指出一个不能由此推出的错误结论。


### Q3

某模型在第一轮 forward/backward 成功，但第一次 optimizer step 后，第二轮训练 OOM。请回答：

1. 哪类显存状态可能是在第一次 optimizer step 附近才完整建立？
2. 为什么"第一轮成功"不能证明后续训练一定能运行？
3. 你至少还需要测量哪两类信息，才能给出可信诊断？

## 检查与通过标准

总分 10 分：

- 代码正确、分量无遗漏：4 分
- 三个问题：每题 2 分
- 通过线：8 分

即使总分达到 8 分，以下任一关键误解仍会导致暂不通过：

- 混淆 GB 和 GiB。
- 把静态账本称为峰值显存。
- 遗漏 master weights 或 Adam 的某个状态。
- 根据单一配置断言某模型"绝对无法训练"。


## 官方主参考

- [Ultra-Scale Playbook](https://huggingface.co/spaces/nanotron/ultrascale-playbook)
- [PyTorch distributed documentation](https://pytorch.org/docs/stable/distributed.html)